# SPINE-GPE v7 — Layout Closure + RAIS Formal Baseline v1.0.0

Fluxo sequencial:

1. fechar ou congelar a pendência documental dos layouts PNADc;
2. auditar e certificar o baseline formal RAIS;
3. criar os locks para o futuro `Phase 0 Master Harmonization & Evidence Lock`.

Revise todas as células de configuração antes da execução.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, subprocess, sys, pandas as pd

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPTS = ROOT / 'scripts'
SCRIPTS.mkdir(parents=True, exist_ok=True)

PACKAGE_DIR = SCRIPTS  # copie os arquivos do pacote para esta pasta
REQ = PACKAGE_DIR / 'requirements_SPINE_GPEv7_LAYOUT_RAIS_PACKAGE_v1.0.0.txt'
LAYOUT_SCRIPT = PACKAGE_DIR / 'SPINE_GPEv7_PNADC_LAYOUT_EQUIVALENCE_CLOSURE_v1.0.0.py'
RAIS_SCRIPT = PACKAGE_DIR / 'SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py'

for p in [REQ, LAYOUT_SCRIPT, RAIS_SCRIPT]:
    assert p.exists(), p
print('ROOT:', ROOT)


Mounted at /content/drive
ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7


In [ ]:
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], text=True, capture_output=True)
print(install.stdout)
print(install.stderr)
assert install.returncode == 0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.8/142.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 19.9 MB/s eta 0:00:00




## 1. Configuração dos layouts PNADc

Adicione pastas que contenham dicionários, inputs ou arquivos de variáveis oficiais. O primeiro audit gera um registry template; copie-o, revise ano e independência documental, e informe o caminho em `LAYOUT_REGISTRY`.


In [ ]:
LAYOUT_YEARS = '2019,2020,2021,2022,2024'
LAYOUT_ROOTS = [
    ROOT / '01_raw' / 'IBGE',
]
LAYOUT_REGISTRY = None  # Ex.: ROOT/'00_admin/documentation/pnadc_layout_source_registry.csv'
HISTORICAL_LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json'

LAYOUT_RUN_ID = 'layout_closure_v100'
cmd = [sys.executable, str(LAYOUT_SCRIPT), '--root', str(ROOT), '--mode', 'audit', '--run-id', LAYOUT_RUN_ID+'_audit', '--years', LAYOUT_YEARS, '--strict']
for r in LAYOUT_ROOTS:
    cmd += ['--layout-root', str(r)]
if HISTORICAL_LOCK.exists():
    cmd += ['--historical-lock', str(HISTORICAL_LOCK)]
audit = subprocess.run(cmd, text=True, capture_output=True)
print(audit.stdout)
print(audit.stderr)
print('exit:', audit.returncode)
assert audit.returncode == 0


2026-07-24 21:33:47,607 | INFO | PNADc Layout Closure v1.0.0 | mode=audit | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
2026-07-24 21:33:47,615 | WARNING | Layout root ausente: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/IBGE
2026-07-24 21:34:46,737 | INFO | Layout closure concluído | status=DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT | lock=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_LAYOUT_EQUIVALENCE_AUDIT_LOCK.json
{
  "run_id": "layout_closure_v100_audit",
  "script_version": "1.0.0",
  "schema_version": "spine-gpe-v7-pnadc-layout-equivalence-closure-1.0.0",
  "mode": "audit",
  "status": "DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT",
  "critical_failures": [],
  "warnings": [
    {
      "test_id": "layout.independent_year_versions",
      "severity": "high",
      "message": "Menos de duas versões anuais independentes foram documentadas.",
      "observed": [],
      "expected": ">=2 annual versions"
    },
    {
   

In [ ]:
from pathlib import Path
import requests

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

DEST = (
    ROOT
    / "00_admin"
    / "documentation"
    / "PNADC"
    / "layouts"
    / "current_official"
)

DEST.mkdir(parents=True, exist_ok=True)

URL = (
    "https://ftp.ibge.gov.br/"
    "Trabalho_e_Rendimento/"
    "Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/"
    "Trimestral/Microdados/Documentacao/"
    "Dicionario_e_input_20221031.zip"
)

OUTPUT = DEST / "Dicionario_e_input_20221031.zip"

with requests.get(URL, stream=True, timeout=180) as response:
    response.raise_for_status()

    with OUTPUT.open("wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

print("Baixado:", OUTPUT)
print("Tamanho:", OUTPUT.stat().st_size, "bytes")

Baixado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/documentation/PNADC/layouts/current_official/Dicionario_e_input_20221031.zip
Tamanho: 70208 bytes


In [ ]:
import zipfile

with zipfile.ZipFile(OUTPUT) as archive:
    for name in archive.namelist():
        print(name)

dicionario_PNADC_microdados_trimestral.xls
input_PNADC_trimestral.sas
input_PNADC_trimestral.txt


In [ ]:
from pathlib import Path
import requests

FILES = {
    "LEIA-ME.pdf": (
        "https://ftp.ibge.gov.br/"
        "Trabalho_e_Rendimento/"
        "Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/"
        "Trimestral/Microdados/LEIA-ME.pdf"
    ),
    "Chaves_PNADC.pdf": (
        "https://ftp.ibge.gov.br/"
        "Trabalho_e_Rendimento/"
        "Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/"
        "Trimestral/Microdados/Documentacao/"
        "Chaves_PNADC.pdf"
    ),
}

for filename, url in FILES.items():
    output = DEST / filename

    with requests.get(url, stream=True, timeout=180) as response:
        response.raise_for_status()

        with output.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    print("Baixado:", output)

Baixado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/documentation/PNADC/layouts/current_official/LEIA-ME.pdf
Baixado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/documentation/PNADC/layouts/current_official/Chaves_PNADC.pdf


In [ ]:
TABLE_LAYOUT = ROOT/'05_outputs/tables/pnadc_layout_equivalence_closure'
registry_templates = sorted(TABLE_LAYOUT.glob('pnadc_layout_source_registry_template_*.csv'))
print('Registry gerado:', registry_templates[-1] if registry_templates else 'não encontrado')
if registry_templates:
    display(pd.read_csv(registry_templates[-1]).head(50))


Registry gerado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_layout_equivalence_closure/pnadc_layout_source_registry_template_layout_closure_v100_audit.csv


EmptyDataError: No columns to parse from file

Revise o registry antes do full. Quando não houver documentos anuais independentes suficientes, o resultado esperado e cientificamente correto é `DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT`.


In [ ]:
cmd = [sys.executable, str(LAYOUT_SCRIPT), '--root', str(ROOT), '--mode', 'full', '--run-id', LAYOUT_RUN_ID, '--years', LAYOUT_YEARS, '--strict']
for r in LAYOUT_ROOTS:
    cmd += ['--layout-root', str(r)]
if LAYOUT_REGISTRY is not None:
    cmd += ['--source-registry', str(LAYOUT_REGISTRY)]
if HISTORICAL_LOCK.exists():
    cmd += ['--historical-lock', str(HISTORICAL_LOCK)]
full_layout = subprocess.run(cmd, text=True, capture_output=True)
print(full_layout.stdout)
print(full_layout.stderr)
print('exit:', full_layout.returncode)
assert full_layout.returncode == 0

layout_lock_path = ROOT/'00_admin/PNADC_LAYOUT_EQUIVALENCE_FINAL_LOCK.json'
layout_lock = json.loads(layout_lock_path.read_text(encoding='utf-8'))
print(json.dumps(layout_lock, ensure_ascii=False, indent=2))
assert layout_lock['status'] in ['LAYOUT_EQUIVALENCE_CONFIRMED','DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT']


## 2. Configuração RAIS

Aponte para as pastas/arquivos de vínculos. Não inclua arquivos de estabelecimentos. Para 2024, forneça o `De-Para Microdados.xlsx` oficial.

O universo principal é CBO `519110` e vínculo ativo em 31/12.


In [ ]:
RAIS_YEARS = '2017,2018,2019,2020,2021,2022,2023,2024'
RAIS_ROOTS = [
    ROOT / '01_raw' / 'MTE' / 'RAIS',
]
RAIS_SOURCE_FILES = []
DEPARA_2024 = ROOT / '00_admin' / 'documentation' / 'De-Para Microdados.xlsx'
DEFLATOR_CSV = None  # arquivo com year,factor_to_base; 2022=1
GOLDEN_CSV = None    # arquivo template do pacote, preenchido com valores oficiais
MINIMUM_WAGE_CSV = None
PRIMARY_CBO = '519110'
RAIS_RUN_ID = 'rais_formal_publication_v100'

cmd = [sys.executable, str(RAIS_SCRIPT), '--root', str(ROOT), '--mode', 'audit', '--run-id', RAIS_RUN_ID+'_audit', '--years', RAIS_YEARS, '--primary-cbo', PRIMARY_CBO]
for r in RAIS_ROOTS:
    cmd += ['--rais-root', str(r)]
for f in RAIS_SOURCE_FILES:
    cmd += ['--source-file', str(f)]
if DEPARA_2024.exists():
    cmd += ['--depara-2024', str(DEPARA_2024)]
rais_audit = subprocess.run(cmd, text=True, capture_output=True)
print(rais_audit.stdout)
print(rais_audit.stderr)
print('exit:', rais_audit.returncode)
assert rais_audit.returncode == 0


In [ ]:
audit_lock_path = ROOT/'00_admin/RAIS_FORMAL_AUDIT_LOCK.json'
rais_audit_lock = json.loads(audit_lock_path.read_text(encoding='utf-8'))
print(json.dumps(rais_audit_lock, ensure_ascii=False, indent=2))
source_audit_path = Path(rais_audit_lock['artifacts']['source_audit'])
display(pd.read_csv(source_audit_path).head(50))


## 3. Execução RAIS completa

O comando abaixo requer pelo menos seis anos certificados. Ajuste `--minimum-years` somente com justificativa documental.


In [ ]:
cmd = [
    sys.executable, str(RAIS_SCRIPT), '--root', str(ROOT), '--mode', 'full',
    '--run-id', RAIS_RUN_ID, '--years', RAIS_YEARS, '--primary-cbo', PRIMARY_CBO,
    '--chunksize', '300000', '--minimum-years', '6', '--real-base-year', '2022', '--strict'
]
for r in RAIS_ROOTS:
    cmd += ['--rais-root', str(r)]
for f in RAIS_SOURCE_FILES:
    cmd += ['--source-file', str(f)]
if DEPARA_2024.exists():
    cmd += ['--depara-2024', str(DEPARA_2024)]
if DEFLATOR_CSV is not None:
    cmd += ['--deflator-csv', str(DEFLATOR_CSV)]
if GOLDEN_CSV is not None:
    cmd += ['--golden-csv', str(GOLDEN_CSV)]
if MINIMUM_WAGE_CSV is not None:
    cmd += ['--minimum-wage-csv', str(MINIMUM_WAGE_CSV)]

rais_full = subprocess.run(cmd, text=True, capture_output=True)
print(rais_full.stdout)
print(rais_full.stderr)
print('exit:', rais_full.returncode)
assert rais_full.returncode == 0


In [ ]:
rais_lock_path = ROOT/'00_admin/RAIS_FORMAL_CERTIFICATION_LOCK.json'
rais_freeze_path = ROOT/'00_admin/RAIS_FORMAL_CORE_FREEZE.json'
rais_lock = json.loads(rais_lock_path.read_text(encoding='utf-8'))
rais_freeze = json.loads(rais_freeze_path.read_text(encoding='utf-8'))
print(json.dumps(rais_lock, ensure_ascii=False, indent=2))
print(json.dumps(rais_freeze, ensure_ascii=False, indent=2))
assert rais_lock['status'] == 'CORE_CERTIFIED'
assert rais_freeze['status'] == 'FROZEN'
assert rais_lock['evidence_tier'] == 'D'
assert rais_lock['platform_direct_observed'] is False


In [ ]:
TABLE_RAIS = ROOT/'05_outputs/tables/rais_formal_certification'
quality = pd.read_csv(rais_lock['artifacts']['quality'])
special = pd.read_csv(rais_lock['artifacts']['special_geographies'])
geography = pd.read_csv(rais_lock['artifacts']['geography'])
demographics = pd.read_csv(rais_lock['artifacts']['demographics'])
print('QUALIDADE')
display(quality)
print('BRASIL / NORDESTE / PE / RECIFE')
display(special)
print('DEMOGRAFIA')
display(demographics.head(100))


## 4. Gates finais

O bloco está pronto para entrar no futuro Master Lock apenas quando:

- o layout closure estiver confirmado ou congelado como limitação documental;
- a RAIS estiver `CORE_CERTIFIED` e `FROZEN`;
- o relatório mantiver a unidade como vínculo formal;
- plataforma direta e informalidade permanecerem explicitamente não observadas.


In [ ]:
assert layout_lock['status'] in ['LAYOUT_EQUIVALENCE_CONFIRMED','DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT']
assert rais_lock['status'] == 'CORE_CERTIFIED'
assert rais_freeze['read_only'] is True
print('LAYOUT CLOSURE + RAIS FORMAL READY FOR PHASE 0 MASTER HARMONIZATION LOCK')


In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

LAYOUT_SCRIPT = (
    ROOT / "scripts"
    / "SPINE_GPEv7_PNADC_LAYOUT_EQUIVALENCE_CLOSURE_v1.0.1.py"
)

HISTORICAL_LOCK = (
    ROOT / "00_admin"
    / "PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json"
)

EMPTY_LAYOUT_ROOT = (
    ROOT / "00_admin"
    / "documentation"
    / "PNADC"
    / "layouts_not_recovered"
)

EMPTY_LAYOUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Script:", LAYOUT_SCRIPT.exists(), LAYOUT_SCRIPT)
print("Lock histórico:", HISTORICAL_LOCK.exists(), HISTORICAL_LOCK)
print("Pasta documental:", EMPTY_LAYOUT_ROOT)

Script: False /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_LAYOUT_EQUIVALENCE_CLOSURE_v1.0.1.py
Lock histórico: True /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json
Pasta documental: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/documentation/PNADC/layouts_not_recovered


In [ ]:
import subprocess
import sys

LAYOUT_RUN_ID = "layout_doclimited_final_v101"

cmd_layout = [
    sys.executable,
    str(LAYOUT_SCRIPT),
    "--root", str(ROOT),
    "--mode", "full",
    "--run-id", LAYOUT_RUN_ID,
    "--years", "2019,2020,2021,2022,2024",
    "--layout-root", str(EMPTY_LAYOUT_ROOT),
    "--historical-lock", str(HISTORICAL_LOCK),
    "--strict",
]

layout_run = subprocess.run(
    cmd_layout,
    text=True,
    capture_output=True,
    check=False,
)

print(layout_run.stdout)
print(layout_run.stderr)
print("Exit code:", layout_run.returncode)

assert layout_run.returncode == 0


/usr/bin/python3: can't open file '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_LAYOUT_EQUIVALENCE_CLOSURE_v1.0.1.py': [Errno 2] No such file or directory

Exit code: 2


AssertionError: 

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib
import shutil
import zipfile

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7/raw_RAIS"
)

SCRIPTS = ROOT / "scripts"
SCRIPTS.mkdir(parents=True, exist_ok=True)

ZIP_NAME = "SPINE_GPEv7_LAYOUT_RAIS_PACKAGE_v1.0.1.zip"

# Procura o pacote no Drive e na sessão atual do Colab.
zip_candidates = list(
    Path("/content/drive/MyDrive").rglob(ZIP_NAME)
)

zip_candidates += list(
    Path("/content").glob(ZIP_NAME)
)

if not zip_candidates:
    print(
        "O ZIP não foi localizado no Drive. "
        "Selecione-o agora no seu computador."
    )

    uploaded = files.upload()

    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(
            f"Era esperado o arquivo {ZIP_NAME}"
        )

    zip_path = Path("/content") / ZIP_NAME
else:
    # Usa a cópia mais recentemente modificada.
    zip_path = max(
        zip_candidates,
        key=lambda p: p.stat().st_mtime,
    )

print("Pacote localizado em:", zip_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pacote localizado em: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/raw_RAIS/SPINE_GPEv7_LAYOUT_RAIS_PACKAGE_v1.0.1.zip


In [ ]:
import hashlib

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


LAYOUT_SCRIPT = layout_target
RAIS_SCRIPT = rais_target

layout_hash = sha256_file(LAYOUT_SCRIPT)

print("Layout:", LAYOUT_SCRIPT)
print("SHA-256:", layout_hash)

EXPECTED_LAYOUT_HASH = (
    "a7769c62ec189dfeed5ece308ee12bf324233e27dabe9b2958f0537f56229509"
)

assert layout_hash == EXPECTED_LAYOUT_HASH, (
    "O engine de layout não corresponde à v1.0.1 esperada."
)

print("Engine v1.0.1 íntegro e pronto.")

NameError: name 'layout_target' is not defined

In [ ]:
from pathlib import Path
import hashlib

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

SCRIPTS = ROOT / "scripts"

LAYOUT_SCRIPT = (
    SCRIPTS
    / "SPINE_GPEv7_PNADC_LAYOUT_"
      "EQUIVALENCE_CLOSURE_v1.0.1.py"
)

RAIS_SCRIPT = (
    SCRIPTS
    / "SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py"
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


print("Engine de layout:", LAYOUT_SCRIPT)
print("Existe:", LAYOUT_SCRIPT.is_file())

print("\nEngine RAIS:", RAIS_SCRIPT)
print("Existe:", RAIS_SCRIPT.is_file())

Engine de layout: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_LAYOUT_EQUIVALENCE_CLOSURE_v1.0.1.py
Existe: True

Engine RAIS: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py
Existe: True


In [ ]:
EXPECTED_LAYOUT_HASH = (
    "a7769c62ec189dfeed5ece308ee12bf324233e27dabe9b2958f0537f56229509"
)

layout_hash = sha256_file(LAYOUT_SCRIPT)

print("SHA-256 encontrado:", layout_hash)
print("SHA-256 esperado:  ", EXPECTED_LAYOUT_HASH)

assert layout_hash == EXPECTED_LAYOUT_HASH, (
    "O engine de layout existente não corresponde à v1.0.1."
)

print("\nEngine de layout v1.0.1 íntegro e pronto.")

SHA-256 encontrado: a7769c62ec189dfeed5ece308ee12bf324233e27dabe9b2958f0537f56229509
SHA-256 esperado:   a7769c62ec189dfeed5ece308ee12bf324233e27dabe9b2958f0537f56229509

Engine de layout v1.0.1 íntegro e pronto.


In [ ]:
import subprocess
import sys

HISTORICAL_LOCK = (
    ROOT
    / "00_admin"
    / "PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json"
)

EMPTY_LAYOUT_ROOT = (
    ROOT
    / "00_admin"
    / "documentation"
    / "PNADC"
    / "layouts_not_recovered"
)

EMPTY_LAYOUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LAYOUT_RUN_ID = "layout_doclimited_final_v101"

assert LAYOUT_SCRIPT.is_file()
assert HISTORICAL_LOCK.is_file(), (
    f"Lock histórico não encontrado: {HISTORICAL_LOCK}"
)

cmd_layout = [
    sys.executable,
    str(LAYOUT_SCRIPT),
    "--root", str(ROOT),
    "--mode", "full",
    "--run-id", LAYOUT_RUN_ID,
    "--years", "2019,2020,2021,2022,2024",
    "--layout-root", str(EMPTY_LAYOUT_ROOT),
    "--historical-lock", str(HISTORICAL_LOCK),
    "--strict",
]

layout_run = subprocess.run(
    cmd_layout,
    text=True,
    capture_output=True,
    check=False,
)

print("STDOUT:\n", layout_run.stdout)
print("\nSTDERR:\n", layout_run.stderr)
print("\nExit code:", layout_run.returncode)

assert layout_run.returncode == 0, (
    "A execução encontrou um erro. "
    "Leia o STDERR exibido acima."
)

STDOUT:
 2026-07-25 01:52:56,597 | INFO | PNADc Layout Closure v1.0.1 | mode=full | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
2026-07-25 01:52:56,607 | WARNING | Nenhum documento candidato de layout foi localizado nas roots: ['/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/documentation/PNADC/layouts_not_recovered']. O registry será gerado com placeholders e o status não poderá ser CONFIRMED.
2026-07-25 01:53:40,190 | INFO | Layout closure concluído | status=DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT | lock=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_LAYOUT_EQUIVALENCE_FINAL_LOCK.json
{
  "run_id": "layout_doclimited_final_v101",
  "script_version": "1.0.1",
  "schema_version": "spine-gpe-v7-pnadc-layout-equivalence-closure-1.0.1",
  "mode": "full",
  "status": "DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT",
  "critical_failures": [],
  "warnings": [
    {
      "test_id": "layout.independent_year_versions",
      "severit

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

FINAL_LAYOUT_LOCK = (
    ROOT
    / "00_admin"
    / "PNADC_LAYOUT_EQUIVALENCE_FINAL_LOCK.json"
)

layout_lock = json.loads(
    FINAL_LAYOUT_LOCK.read_text(encoding="utf-8")
)

assert (
    layout_lock["status"]
    == "DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT"
)
assert layout_lock["critical_failures"] == []

operational_path = Path(
    layout_lock["artifacts"]["operational"]
)

assert operational_path.is_file(), operational_path

operational = pd.read_csv(operational_path)

print("Arquivo:", operational_path)
print("Dimensão:", operational.shape)
print("Colunas:", operational.columns.tolist())

display(operational.head(50))

Arquivo: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_layout_equivalence_closure/pnadc_operational_domain_stability_layout_doclimited_final_v101.csv
Dimensão: (98, 13)
Colunas: ['source_period', 'year', 'variable', 'source_path', 'source_sha256', 'parquet_column', 'dtype', 'n', 'missing_rate', 'n_unique', 'min', 'max', 'sample_domain_signature']


,source_period,year,variable,source_path,source_sha256,parquet_column,dtype,n,missing_rate,n_unique,min,max,sample_domain_signature
0,2022q4,2022,V4010,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,2c27345add80ffedcc7e7ecbaa4de8a587b22f08707048...,occupation_code,string,178163,0.000000,415,0.000000,9629.000000,e7eff969d47ebfd970ffb4109b19245b324db5f02aee13...
1,2022q4,2022,V4013,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,2c27345add80ffedcc7e7ecbaa4de8a587b22f08707048...,activity_code,string,178163,0.000000,216,0.000000,99000.000000,58ba47b4a6d48a802ab5f796b44ab4811d9e9afdbf193c...
2,2022q4,2022,VD4009,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,2c27345add80ffedcc7e7ecbaa4de8a587b22f08707048...,position_code,string,178163,0.000000,7,1.000000,9.000000,193f6491ce445e0d24ddc3657fd44376832ff00b090607...
3,2022q4,2022,V1028,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,2c27345add80ffedcc7e7ecbaa4de8a587b22f08707048...,survey_weight,Float64,178163,0.000000,113594,8.502298,18229.617642,970b89dc46021e3f0356936e7d1133a81713ce640c20be...
4,2022q4,2022,UF,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,2c27345add80ffedcc7e7ecbaa4de8a587b22f08707048...,UF,string,178163,0.000000,27,11.000000,53.000000,1b7d508e1fd83dffd1235ab019b25b7a4a9131acebdda6...
5,2022q4,2022,Capital,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,2c27345add80ffedcc7e7ecbaa4de8a587b22f08707048...,Capital,string,178163,0.751048,27,11.000000,53.000000,1b7d508e1fd83dffd1235ab019b25b7a4a9131acebdda6...
6,2022q4,2022,RM_RIDE,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,2c27345add80ffedcc7e7ecbaa4de8a587b22f08707048...,RM_RIDE,string,178163,0.668259,21,13.000000,52.000000,8498d4105f30e5b792203d5f66ec0081f5dedb63b19989...
7,2024q3,2024,V4010,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,988a2cab01b28d3ddb515ae659657bb91bb37a192308a4...,occupation_code,string,184157,0.000000,416,0.000000,9629.000000,9deb4ef9d1c7a317477e9e749896e27345e30485f70066...
8,2024q3,2024,V4013,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,988a2cab01b28d3ddb515ae659657bb91bb37a192308a4...,activity_code,string,184157,0.000000,216,0.000000,99000.000000,58ba47b4a6d48a802ab5f796b44ab4811d9e9afdbf193c...
9,2024q3,2024,VD4009,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,988a2cab01b28d3ddb515ae659657bb91bb37a192308a4...,position_code,string,184157,0.000000,7,1.000000,9.000000,193f6491ce445e0d24ddc3657fd44376832ff00b090607...


In [ ]:
CORE_VARS = {
    "V4010",
    "V4013",
    "VD4009",
    "V1028",
    "Estrato",
    "UPA",
}

EXPECTED_PERIODS = {
    f"{year}q{quarter}"
    for year in [2019, 2020, 2021]
    for quarter in [1, 2, 3, 4]
}

period_candidates = [
    "source_period",
    "period",
    "periodo",
    "year_quarter",
]

variable_candidates = [
    "variable",
    "variavel",
    "column",
    "column_name",
]

period_col = next(
    (
        column
        for column in period_candidates
        if column in operational.columns
    ),
    None,
)

variable_col = next(
    (
        column
        for column in variable_candidates
        if column in operational.columns
    ),
    None,
)

assert not operational.empty, (
    "A tabela operacional está vazia. "
    "Não avance para a RAIS antes de investigar."
)

assert period_col is not None, (
    "Não foi identificada a coluna de período. "
    f"Colunas encontradas: {operational.columns.tolist()}"
)

assert variable_col is not None, (
    "Não foi identificada a coluna de variável. "
    f"Colunas encontradas: {operational.columns.tolist()}"
)

observed_periods = set(
    operational[period_col]
    .astype(str)
    .str.lower()
    .str.replace("t", "q", regex=False)
)

observed_vars = set(
    operational[variable_col].astype(str)
)

missing_periods = EXPECTED_PERIODS - observed_periods
missing_vars = CORE_VARS - observed_vars

print("Coluna de período:", period_col)
print("Coluna de variável:", variable_col)
print("Períodos ausentes:", sorted(missing_periods))
print("Variáveis ausentes:", sorted(missing_vars))

assert not missing_periods, (
    f"Períodos históricos ausentes: {sorted(missing_periods)}"
)

assert not missing_vars, (
    f"Variáveis centrais ausentes: {sorted(missing_vars)}"
)

print(
    "\nLAYOUT DOCUMENTATION LIMITED, "
    "OPERATIONAL CONSISTENCY VERIFIED"
)

Coluna de período: source_period
Coluna de variável: variable
Períodos ausentes: []
Variáveis ausentes: ['Estrato', 'UPA']


AssertionError: Variáveis centrais ausentes: ['Estrato', 'UPA']

In [ ]:
import pandas as pd

REQUIRED_MODEL_VARS = {
    "V4010",
    "V4013",
    "VD4009",
    "V1028",
}

REQUIRED_HISTORICAL_PERIODS = {
    f"{year}q{quarter}"
    for year in [2019, 2020, 2021]
    for quarter in [1, 2, 3, 4]
}

period_col = "source_period"
variable_col = "variable"

operational_check = operational.copy()

operational_check[period_col] = (
    operational_check[period_col]
    .astype(str)
    .str.lower()
    .str.replace("t", "q", regex=False)
)

operational_check[variable_col] = (
    operational_check[variable_col]
    .astype(str)
    .str.strip()
)

historical_operational = operational_check[
    operational_check[period_col].isin(
        REQUIRED_HISTORICAL_PERIODS
    )
].copy()

assert not historical_operational.empty, (
    "Nenhuma observação operacional histórica foi encontrada."
)

observed_periods = set(
    historical_operational[period_col]
)

missing_periods = (
    REQUIRED_HISTORICAL_PERIODS
    - observed_periods
)

assert not missing_periods, (
    f"Períodos históricos ausentes: "
    f"{sorted(missing_periods)}"
)

coverage = pd.crosstab(
    historical_operational[period_col],
    historical_operational[variable_col],
)

missing_cells = []

for period in sorted(REQUIRED_HISTORICAL_PERIODS):
    for variable in sorted(REQUIRED_MODEL_VARS):
        if (
            period not in coverage.index
            or variable not in coverage.columns
            or coverage.loc[period, variable] == 0
        ):
            missing_cells.append(
                {
                    "source_period": period,
                    "variable": variable,
                }
            )

assert not missing_cells, (
    "Há células período × variável ausentes: "
    f"{missing_cells}"
)

core_rows = historical_operational[
    historical_operational[variable_col].isin(
        REQUIRED_MODEL_VARS
    )
].copy()

core_rows["missing_rate"] = pd.to_numeric(
    core_rows["missing_rate"],
    errors="coerce",
)

invalid_missing = core_rows[
    core_rows["missing_rate"].isna()
    | (core_rows["missing_rate"] > 0.001)
]

assert invalid_missing.empty, (
    "Há missing inesperado nas variáveis centrais:\n"
    f"{invalid_missing[[period_col, variable_col, 'missing_rate']]}"
)

print("Períodos históricos verificados:", len(observed_periods))
print("Variáveis centrais:", sorted(REQUIRED_MODEL_VARS))
print("Células período × variável:", len(core_rows))
print(
    "Missing máximo:",
    core_rows["missing_rate"].max(),
)

print(
    "\nLAYOUT DOCUMENTATION LIMITED, "
    "OPERATIONAL MODEL CORE VERIFIED"
)

Períodos históricos verificados: 12
Variáveis centrais: ['V1028', 'V4010', 'V4013', 'VD4009']
Células período × variável: 48
Missing máximo: 0.0

LAYOUT DOCUMENTATION LIMITED, OPERATIONAL MODEL CORE VERIFIED


In [ ]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import pyarrow.parquet as pq


def normalize_name(value: str) -> str:
    value = unicodedata.normalize(
        "NFKD",
        str(value),
    )

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(char)
    )

    return re.sub(
        r"[^a-z0-9]",
        "",
        value.lower(),
    )


DESIGN_ALIASES = {
    "Estrato": {
        "estrato",
        "stratum",
        "surveystratum",
        "designstratum",
        "samplingstratum",
    },
    "UPA": {
        "upa",
        "psu",
        "surveypsu",
        "primarysamplingunit",
        "cluster",
        "samplingcluster",
    },
}

source_period_paths = (
    historical_operational[
        ["source_period", "source_path"]
    ]
    .drop_duplicates()
    .sort_values("source_period")
)

design_rows = []

for row in source_period_paths.itertuples(index=False):
    source_period = row.source_period
    source_path = Path(row.source_path)

    if not source_path.is_file():
        design_rows.append(
            {
                "source_period": source_period,
                "source_path": str(source_path),
                "schema_read": False,
                "Estrato": None,
                "UPA": None,
                "notes": "Arquivo não localizado.",
            }
        )
        continue

    try:
        schema_names = (
            pq.ParquetFile(source_path)
            .schema_arrow
            .names
        )

        normalized_schema = {
            normalize_name(column): column
            for column in schema_names
        }

        result = {
            "source_period": source_period,
            "source_path": str(source_path),
            "schema_read": True,
            "notes": "",
        }

        for canonical, aliases in DESIGN_ALIASES.items():
            matched = [
                original
                for normalized, original
                in normalized_schema.items()
                if normalized in aliases
            ]

            result[canonical] = (
                matched[0]
                if matched
                else None
            )

        design_rows.append(result)

    except Exception as error:
        design_rows.append(
            {
                "source_period": source_period,
                "source_path": str(source_path),
                "schema_read": False,
                "Estrato": None,
                "UPA": None,
                "notes": repr(error),
            }
        )

design_audit = pd.DataFrame(design_rows)

display(
    design_audit[
        [
            "source_period",
            "schema_read",
            "Estrato",
            "UPA",
            "notes",
        ]
    ]
)

print("\nCobertura:")
print(
    design_audit[
        ["Estrato", "UPA"]
    ].notna().sum()
)

,source_period,schema_read,Estrato,UPA,notes
0,2019q1,True,survey_stratum,survey_psu,
1,2019q2,True,survey_stratum,survey_psu,
2,2019q3,True,survey_stratum,survey_psu,
3,2019q4,True,survey_stratum,survey_psu,
4,2020q1,True,survey_stratum,survey_psu,
5,2020q2,True,survey_stratum,survey_psu,
6,2020q3,True,survey_stratum,survey_psu,
7,2020q4,True,survey_stratum,survey_psu,
8,2021q1,True,survey_stratum,survey_psu,
9,2021q2,True,survey_stratum,survey_psu,



Cobertura:
Estrato    12
UPA        12
dtype: int64


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


design_coverage = {
    variable: int(
        design_audit[variable].notna().sum()
    )
    for variable in ["Estrato", "UPA"]
}

design_complete = all(
    value == len(REQUIRED_HISTORICAL_PERIODS)
    for value in design_coverage.values()
)

ADJUDICATION_PATH = (
    ROOT
    / "00_admin"
    / "PNADC_LAYOUT_CLOSURE_ADJUDICATION.json"
)

adjudication = {
    "run_id": layout_lock["run_id"],
    "status": (
        "DOCUMENTATION_LIMITED_"
        "OPERATIONALLY_CONSISTENT"
    ),
    "layout_lock": str(FINAL_LAYOUT_LOCK),
    "layout_lock_sha256": sha256_file(
        FINAL_LAYOUT_LOCK
    ),
    "operational_artifact": str(
        operational_path
    ),
    "operational_artifact_sha256": sha256_file(
        operational_path
    ),
    "historical_periods_expected": sorted(
        REQUIRED_HISTORICAL_PERIODS
    ),
    "historical_periods_verified": sorted(
        observed_periods
    ),
    "model_core_variables_verified": sorted(
        REQUIRED_MODEL_VARS
    ),
    "model_core_period_variable_cells": int(
        len(core_rows)
    ),
    "model_core_missing_rate_max": float(
        core_rows["missing_rate"].max()
    ),
    "design_variable_schema_coverage": (
        design_coverage
    ),
    "design_variables_complete_in_parquet_schema": (
        design_complete
    ),
    "decision": (
        "A equivalência documental anual não foi "
        "demonstrada. A consistência operacional "
        "do núcleo analítico V4010, V4013, VD4009 "
        "e V1028 foi verificada nos 12 trimestres "
        "históricos. Estrato e UPA foram tratados "
        "como identificadores de desenho amostral "
        "e auditados separadamente, sem serem "
        "confundidos com preditores do backcast."
    ),
    "claim_ceiling": layout_lock[
        "claim_ceiling"
    ],
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

ADJUDICATION_PATH.write_text(
    json.dumps(
        adjudication,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Adjudicação:", ADJUDICATION_PATH)
print(
    json.dumps(
        adjudication,
        ensure_ascii=False,
        indent=2,
    )
)

Adjudicação: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_LAYOUT_CLOSURE_ADJUDICATION.json
{
  "run_id": "layout_doclimited_final_v101",
  "status": "DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT",
  "layout_lock": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PNADC_LAYOUT_EQUIVALENCE_FINAL_LOCK.json",
  "layout_lock_sha256": "caf015ee57f2f20648aa0fb14990829c54cb82106130b0fd3b8abf11987edb3f",
  "operational_artifact": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/pnadc_layout_equivalence_closure/pnadc_operational_domain_stability_layout_doclimited_final_v101.csv",
  "operational_artifact_sha256": "1927bbf368f11f86dfb63b661299778fbb2f7b0ad75a9d833570b8bc94a1ffb2",
  "historical_periods_expected": [
    "2019q1",
    "2019q2",
    "2019q3",
    "2019q4",
    "2020q1",
    "2020q2",
    "2020q3",
    "2020q4",
    "2021q1",
    "2021q2",
    "2021q3",
    "2021q4"
  ],
  "historical_periods_verified": [
    "2019q1",
  

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

RAIS_ROOT = (
    ROOT
    / "01_raw"
    / "MTE"
    / "RAIS"
)

SUPPORTED_RAIS = {
    ".7z",
    ".zip",
    ".txt",
    ".csv",
    ".comt",
    ".parquet",
}

RAIS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

rais_files = sorted(
    path
    for path in RAIS_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in SUPPORTED_RAIS
)

print("RAIS root:", RAIS_ROOT)
print("Pasta existe:", RAIS_ROOT.exists())
print("Arquivos encontrados:", len(rais_files))

for path in rais_files[:200]:
    print(path.relative_to(RAIS_ROOT))

RAIS root: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS
Pasta existe: True
Arquivos encontrados: 0


In [ ]:
RAIS_SCRIPT = (
    ROOT
    / "scripts"
    / "SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py"
)

RAIS_DOC = (
    ROOT
    / "00_admin"
    / "documentation"
    / "RAIS"
)

RAIS_DOC.mkdir(
    parents=True,
    exist_ok=True,
)

DEPARA_2024 = (
    RAIS_DOC
    / "De-Para Microdados.xlsx"
)

print("Engine RAIS:", RAIS_SCRIPT)
print("Engine existe:", RAIS_SCRIPT.is_file())

print("\nDe-para 2024:", DEPARA_2024)
print("De-para existe:", DEPARA_2024.is_file())

assert RAIS_SCRIPT.is_file(), (
    f"Engine RAIS não encontrado: {RAIS_SCRIPT}"
)

Engine RAIS: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py
Engine existe: True

De-para 2024: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/documentation/RAIS/De-Para Microdados.xlsx
De-para existe: False


In [ ]:
import subprocess
import sys

assert rais_files, (
    "Nenhum microdado RAIS foi encontrado em "
    f"{RAIS_ROOT}. Coloque os arquivos antes do audit."
)

RAIS_AUDIT_RUN_ID = (
    "rais_formal_2019_2024_audit_v100"
)

cmd_rais_audit = [
    sys.executable,
    str(RAIS_SCRIPT),
    "--root",
    str(ROOT),
    "--mode",
    "audit",
    "--run-id",
    RAIS_AUDIT_RUN_ID,
    "--years",
    "2019,2020,2021,2022,2023,2024",
    "--rais-root",
    str(RAIS_ROOT),
    "--primary-cbo",
    "519110",
    "--strict",
]

if DEPARA_2024.is_file():
    cmd_rais_audit.extend([
        "--depara-2024",
        str(DEPARA_2024),
    ])

print("Executando:\n")
print(" ".join(cmd_rais_audit))

rais_audit_run = subprocess.run(
    cmd_rais_audit,
    text=True,
    capture_output=True,
    check=False,
)

print("\nSTDOUT:\n")
print(rais_audit_run.stdout)

print("\nSTDERR:\n")
print(rais_audit_run.stderr)

print(
    "\nExit code:",
    rais_audit_run.returncode,
)

AssertionError: Nenhum microdado RAIS foi encontrado em /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS. Coloque os arquivos antes do audit.

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

candidates = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if (
        "rais_vinc" in name
        or "rais_estab" in name
        or "rais2022" in name
        or "rais_2022" in name
    ):
        candidates.append(path)

print("Arquivos RAIS encontrados:", len(candidates))

for path in candidates[:200]:
    print(path)

Arquivos RAIS encontrados: 0


In [ ]:
from pathlib import Path
import subprocess
import re

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

RAIS_2022_DIR = (
    ROOT
    / "01_raw"
    / "MTE"
    / "RAIS"
    / "2022"
)

RAIS_2022_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FTP_2022 = (
    "ftp://ftp.mtps.gov.br/"
    "pdet/microdados/RAIS/2022/"
)

result = subprocess.run(
    [
        "curl",
        "--ftp-pasv",
        "--list-only",
        "--silent",
        "--show-error",
        FTP_2022,
    ],
    text=True,
    capture_output=True,
    check=False,
)

print("Exit code:", result.returncode)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

listing = [
    line.strip()
    for line in result.stdout.splitlines()
    if line.strip()
]

print("\nArquivos disponíveis:", len(listing))

for name in listing:
    print(name)

Exit code: 0

Arquivos disponíveis: 8
RAIS_ESTAB_PUB.7z
RAIS_VINC_PUB_CENTRO_OESTE.7z
RAIS_VINC_PUB_MG_ES_RJ.7z
RAIS_VINC_PUB_NI.7z
RAIS_VINC_PUB_NORDESTE.7z
RAIS_VINC_PUB_NORTE.7z
RAIS_VINC_PUB_SP.7z
RAIS_VINC_PUB_SUL.7z


In [ ]:
worker_archives = sorted(
    {
        line.split("/")[-1]
        for line in listing
        if re.search(
            r"rais.*vinc.*\.(7z|zip)$",
            line,
            flags=re.IGNORECASE,
        )
    }
)

print(
    "Arquivos de vínculos selecionados:",
    len(worker_archives),
)

for filename in worker_archives:
    print(filename)

assert worker_archives, (
    "Nenhum arquivo de vínculos foi identificado "
    "na listagem oficial."
)

Arquivos de vínculos selecionados: 7
RAIS_VINC_PUB_CENTRO_OESTE.7z
RAIS_VINC_PUB_MG_ES_RJ.7z
RAIS_VINC_PUB_NI.7z
RAIS_VINC_PUB_NORDESTE.7z
RAIS_VINC_PUB_NORTE.7z
RAIS_VINC_PUB_SP.7z
RAIS_VINC_PUB_SUL.7z


In [ ]:
from urllib.parse import quote
import subprocess

download_results = []

for filename in worker_archives:
    destination = RAIS_2022_DIR / filename

    url = (
        FTP_2022
        + quote(filename)
    )

    print("\n" + "=" * 80)
    print("Arquivo:", filename)
    print("Destino:", destination)

    command = [
        "curl",
        "--ftp-pasv",
        "--fail",
        "--show-error",
        "--retry", "10",
        "--retry-delay", "10",
        "--continue-at", "-",
        "--output", str(destination),
        url,
    ]

    completed = subprocess.run(
        command,
        text=True,
        check=False,
    )

    download_results.append(
        {
            "filename": filename,
            "exit_code": completed.returncode,
            "exists": destination.is_file(),
            "size_bytes": (
                destination.stat().st_size
                if destination.is_file()
                else 0
            ),
        }
    )

    if completed.returncode != 0:
        print(
            "Download interrompido. Execute novamente "
            "esta célula para continuar."
        )

download_results



Arquivo: RAIS_VINC_PUB_CENTRO_OESTE.7z
Destino: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_CENTRO_OESTE.7z

Arquivo: RAIS_VINC_PUB_MG_ES_RJ.7z
Destino: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_MG_ES_RJ.7z

Arquivo: RAIS_VINC_PUB_NI.7z
Destino: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_NI.7z

Arquivo: RAIS_VINC_PUB_NORDESTE.7z
Destino: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_NORDESTE.7z

Arquivo: RAIS_VINC_PUB_NORTE.7z
Destino: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_NORTE.7z

Arquivo: RAIS_VINC_PUB_SP.7z
Destino: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_SP.7z

Arquivo: RAIS_VINC_PUB_SUL.7z
Destino: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_SUL.7z


[{'filename': 'RAIS_VINC_PUB_CENTRO_OESTE.7z',
  'exit_code': 0,
  'exists': True,
  'size_bytes': 279182629},
 {'filename': 'RAIS_VINC_PUB_MG_ES_RJ.7z',
  'exit_code': 0,
  'exists': True,
  'size_bytes': 628291547},
 {'filename': 'RAIS_VINC_PUB_NI.7z',
  'exit_code': 0,
  'exists': True,
  'size_bytes': 63071},
 {'filename': 'RAIS_VINC_PUB_NORDESTE.7z',
  'exit_code': 0,
  'exists': True,
  'size_bytes': 462393275},
 {'filename': 'RAIS_VINC_PUB_NORTE.7z',
  'exit_code': 0,
  'exists': True,
  'size_bytes': 156214163},
 {'filename': 'RAIS_VINC_PUB_SP.7z',
  'exit_code': 0,
  'exists': True,
  'size_bytes': 947704550},
 {'filename': 'RAIS_VINC_PUB_SUL.7z',
  'exit_code': 0,
  'exists': True,
  'size_bytes': 599471446}]

In [ ]:
import pandas as pd

download_audit = pd.DataFrame(
    download_results
)

download_audit["size_mb"] = (
    download_audit["size_bytes"]
    / 1024**2
)

display(download_audit)

failed = download_audit[
    (download_audit["exit_code"] != 0)
    | (~download_audit["exists"])
    | (download_audit["size_bytes"] == 0)
]

assert failed.empty, (
    "Há downloads incompletos. "
    "Execute novamente a célula de download."
)

print(
    "\nTodos os arquivos selecionados "
    "foram baixados."
)

,filename,exit_code,exists,size_bytes,size_mb
0,RAIS_VINC_PUB_CENTRO_OESTE.7z,0,True,279182629,266.249303
1,RAIS_VINC_PUB_MG_ES_RJ.7z,0,True,628291547,599.185512
2,RAIS_VINC_PUB_NI.7z,0,True,63071,0.060149
3,RAIS_VINC_PUB_NORDESTE.7z,0,True,462393275,440.972590
4,RAIS_VINC_PUB_NORTE.7z,0,True,156214163,148.977435
5,RAIS_VINC_PUB_SP.7z,0,True,947704550,903.801489
6,RAIS_VINC_PUB_SUL.7z,0,True,599471446,571.700521



Todos os arquivos selecionados foram baixados.


In [ ]:
from pathlib import Path
import subprocess

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

RAIS_2022_DIR = (
    ROOT
    / "01_raw"
    / "MTE"
    / "RAIS"
    / "2022"
)

NORDESTE_ARCHIVE = (
    RAIS_2022_DIR
    / "RAIS_VINC_PUB_NORDESTE.7z"
)

assert NORDESTE_ARCHIVE.is_file(), NORDESTE_ARCHIVE

# Instala o utilitário somente se não estiver disponível.
install = subprocess.run(
    [
        "bash",
        "-lc",
        (
            "command -v 7z >/dev/null 2>&1 "
            "|| (apt-get -qq update "
            "&& apt-get -qq install -y p7zip-full)"
        ),
    ],
    check=False,
)

assert install.returncode == 0

print("Testando:", NORDESTE_ARCHIVE)
print("Essa etapa pode levar alguns minutos.\n")

integrity = subprocess.run(
    [
        "7z",
        "t",
        "-bd",
        "-y",
        str(NORDESTE_ARCHIVE),
    ],
    text=True,
    check=False,
)

print("\nExit code:", integrity.returncode)

assert integrity.returncode == 0, (
    "O arquivo do Nordeste não passou no teste de integridade."
)

print("Arquivo RAIS Nordeste íntegro.")

Testando: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_NORDESTE.7z
Essa etapa pode levar alguns minutos.


Exit code: 0
Arquivo RAIS Nordeste íntegro.


In [ ]:
listing = subprocess.run(
    [
        "7z",
        "l",
        "-ba",
        str(NORDESTE_ARCHIVE),
    ],
    text=True,
    capture_output=True,
    check=False,
)

print(listing.stdout[:10000])

assert listing.returncode == 0

2024-05-01 05:09:12 ....A   6629461968    462393113  RAIS_VINC_PUB_NORDESTE.txt



In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("py7zr") is None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "py7zr",
        ],
        check=True,
    )

import py7zr

print("py7zr instalado:", py7zr.__version__)

py7zr instalado: 1.1.3


In [ ]:
from pathlib import Path
import shutil

PILOT_ROOT = Path(
    "/content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT"
)

if PILOT_ROOT.exists():
    shutil.rmtree(PILOT_ROOT)

PILOT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PILOT_EMPTY_ROOT = (
    PILOT_ROOT
    / "empty_rais_root"
)

PILOT_EMPTY_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RAIS_SCRIPT = (
    ROOT
    / "scripts"
    / "SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py"
)

assert RAIS_SCRIPT.is_file(), RAIS_SCRIPT
assert NORDESTE_ARCHIVE.is_file(), NORDESTE_ARCHIVE

print("Engine:", RAIS_SCRIPT)
print("Fonte piloto:", NORDESTE_ARCHIVE)
print("Saída temporária:", PILOT_ROOT)

Engine: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py
Fonte piloto: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_NORDESTE.7z
Saída temporária: /content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT


In [ ]:
import subprocess
import sys

PILOT_RUN_ID = "rais_formal_2022_nordeste_pilot_v100"

cmd_pilot = [
    sys.executable,
    str(RAIS_SCRIPT),
    "--root",
    str(PILOT_ROOT),
    "--mode",
    "audit",
    "--run-id",
    PILOT_RUN_ID,
    "--years",
    "2022",
    "--rais-root",
    str(PILOT_EMPTY_ROOT),
    "--source-file",
    str(NORDESTE_ARCHIVE),
    "--primary-cbo",
    "519110",
    "--chunksize",
    "300000",
]

print("Executando:\n")
print(" ".join(cmd_pilot))
print()

process = subprocess.Popen(
    cmd_pilot,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(line, end="")

pilot_exit_code = process.wait()

print("\nPilot exit code:", pilot_exit_code)

Executando:

/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py --root /content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT --mode audit --run-id rais_formal_2022_nordeste_pilot_v100 --years 2022 --rais-root /content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT/empty_rais_root --source-file /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_NORDESTE.7z --primary-cbo 519110 --chunksize 300000

2026-07-25 06:49:03,395 | INFO | RAIS Formal Certifier v1.0.0 | mode=audit | root=/content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT
2026-07-25 06:49:05,691 | INFO | Extraindo /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_NORDESTE.7z
2026-07-25 06:52:57,754 | INFO | RAIS_VINC_PUB_NORDESTE.txt | chunks=20 | rows=6000000 | target=23522
2026-07-25 06:53:53,885 | INFO | RAIS_VINC_PUB_NORDESTE.txt | chunks=40 | rows=12000000 | target=39671
{
  "run_id": "rais_formal_2022_

In [ ]:
import json
import pandas as pd

PILOT_LOCK = (
    PILOT_ROOT
    / "00_admin"
    / "RAIS_FORMAL_AUDIT_LOCK.json"
)

assert PILOT_LOCK.is_file(), (
    f"Lock não encontrado: {PILOT_LOCK}"
)

pilot_lock = json.loads(
    PILOT_LOCK.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        pilot_lock,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "run_id": "rais_formal_2022_nordeste_pilot_v100",
  "script_version": "1.0.0",
  "schema_version": "spine-gpe-v7-rais-formal-certifier-1.0.0",
  "mode": "audit",
  "status": "AUDIT_PASSED",
  "critical_failures": [],
  "warnings": [],
  "years_requested": [
    2022
  ],
  "primary_cbo_codes": [
    "519110"
  ],
  "sensitivity_cbo_codes": [],
  "rais_roots": [
    "/content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT/empty_rais_root"
  ],
  "sources_discovered": 1,
  "files_expanded": 1,
  "depara_2024": null,
  "artifacts": {
    "inventory": "/content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT/05_outputs/tables/rais_formal_certification/rais_source_inventory_rais_formal_2022_nordeste_pilot_v100.csv",
    "source_audit": "/content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT/05_outputs/tables/rais_formal_certification/rais_source_schema_audit_rais_formal_2022_nordeste_pilot_v100.csv"
  },
  "artifact_hashes": {
    "inventory": "778471b245e9f8c29cb447a98bde42d31e07afc2a97015fa6a834378286a467a",
    "

In [ ]:
PILOT_SCHEMA_PATH = Path(
    pilot_lock["artifacts"]["source_audit"]
)

pilot_schema = pd.read_csv(
    PILOT_SCHEMA_PATH
)

display(pilot_schema)

print("\nStatus das fontes:")
print(
    pilot_schema[
        [
            "inferred_year",
            "rows_total",
            "rows_active_all_occupations",
            "rows_target_all",
            "rows_target_active",
            "status",
        ]
    ].to_string(index=False)
)


,source_path,source_sha256,source_extension,source_size_bytes,inferred_year,encoding,delimiter,header_columns,canonical_mapping,rows_total,rows_active_all_occupations,rows_target_all,rows_target_active,status,notes
0,/content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT/...,396ca7ec2e5ad69462633bfba360c0ce75c4db4eb01ad1...,.txt,6629461968,2022,latin1,;,"['Bairros SP', 'Bairros Fortaleza', 'Bairros R...","{'cbo2002': 'CBO Ocupação 2002', 'active_3112'...",13584961,9777008,40469,27198,PARSED,NaN



Status das fontes:
 inferred_year  rows_total  rows_active_all_occupations  rows_target_all  rows_target_active status
          2022    13584961                      9777008            40469               27198 PARSED


In [ ]:
for row in pilot_schema.itertuples(index=False):
    print("\nFonte:", row.source_path)
    print("Status:", row.status)
    print("Ano:", row.inferred_year)
    print("Linhas totais:", row.rows_total)
    print("CBO-alvo total:", row.rows_target_all)
    print("CBO-alvo ativo:", row.rows_target_active)
    print("Mapeamento:")
    print(row.canonical_mapping)


Fonte: /content/SPINE_GPEv7_RAIS_2022_NORDESTE_PILOT/03_intermediate/rais_formal_certification/rais_formal_2022_nordeste_pilot_v100/extracted_7z/943ca74e2dba77c3/RAIS_VINC_PUB_NORDESTE.txt
Status: PARSED
Ano: 2022
Linhas totais: 13584961
CBO-alvo total: 40469
CBO-alvo ativo: 27198
Mapeamento:
{'cbo2002': 'CBO Ocupação 2002', 'active_3112': 'Vínculo Ativo 31/12', 'municipality_work': 'Mun Trab', 'income_avg_nominal': 'Vl Remun Média Nom', 'income_dec_nominal': 'Vl Remun Dezembro Nom', 'income_avg_sm': 'Vl Remun Média (SM)', 'contract_hours': 'Qtd Hora Contr', 'sex': 'Sexo Trabalhador', 'race': 'Raça Cor', 'education': 'Escolaridade após 2005', 'age': 'Idade', 'age_group': 'Faixa Etária', 'link_type': 'Tipo Vínculo', 'cnae_class': 'CNAE 2.0 Classe', 'admission_type': 'Tipo Admissão', 'establishment_size': 'Tamanho Estabelecimento', 'legal_nature': 'Natureza Jurídica'}


In [ ]:
RAIS_NATIONAL_RUN_ID = "rais_formal_2022_national_audit_v100"

cmd_national = [
    sys.executable,
    str(RAIS_SCRIPT),
    "--root",
    str(ROOT),
    "--mode",
    "audit",
    "--run-id",
    RAIS_NATIONAL_RUN_ID,
    "--years",
    "2022",
    "--rais-root",
    str(RAIS_2022_DIR),
    "--primary-cbo",
    "519110",
    "--chunksize",
    "300000",
]

print("Executando audit nacional:\n")
print(" ".join(cmd_national))
print()

process = subprocess.Popen(
    cmd_national,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(line, end="")

national_exit_code = process.wait()

print("\nNational audit exit code:", national_exit_code)

Executando audit nacional:

/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode audit --run-id rais_formal_2022_national_audit_v100 --years 2022 --rais-root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022 --primary-cbo 519110 --chunksize 300000

2026-07-25 07:46:17,115 | INFO | RAIS Formal Certifier v1.0.0 | mode=audit | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
2026-07-25 07:46:42,371 | INFO | Extraindo /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_CENTRO_OESTE.7z
2026-07-25 07:47:40,442 | INFO | Extraindo /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_MG_ES_RJ.7z
2026-07-25 07:50:06,348 | INFO | Extraindo /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/MTE/RAIS/2022/RAIS_VINC_PUB_NI.7z
2026-07-25 07:50:07,55

In [ ]:
from pathlib import Path
import ast
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

SOURCE_AUDIT_PATH = (
    ROOT
    / "05_outputs"
    / "tables"
    / "rais_formal_certification"
    / (
        "rais_source_schema_audit_"
        "rais_formal_2022_national_audit_v100.csv"
    )
)

assert SOURCE_AUDIT_PATH.is_file(), SOURCE_AUDIT_PATH

source_audit = pd.read_csv(SOURCE_AUDIT_PATH)

print("Fontes auditadas:", len(source_audit))
print("Status:")
print(source_audit["status"].value_counts(dropna=False))

display(
    source_audit[
        [
            "source_path",
            "inferred_year",
            "rows_total",
            "rows_active_all_occupations",
            "rows_target_all",
            "rows_target_active",
            "status",
        ]
    ]
)

AssertionError: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/rais_formal_certification/rais_source_schema_audit_rais_formal_2022_national_audit_v100.csv

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import time
import pandas as pd

drive.mount(
    "/content/drive",
    force_remount=False,
)

os.sync()
time.sleep(2)

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

AUDIT_LOCK_PATH = (
    ROOT
    / "00_admin"
    / "RAIS_FORMAL_AUDIT_LOCK.json"
)

assert AUDIT_LOCK_PATH.is_file(), (
    f"Lock da auditoria não encontrado: {AUDIT_LOCK_PATH}"
)

audit_lock = json.loads(
    AUDIT_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

print("Run ID do lock:", audit_lock.get("run_id"))
print("Status:", audit_lock.get("status"))
print(
    "Critical failures:",
    audit_lock.get("critical_failures"),
)

assert (
    audit_lock.get("run_id")
    == "rais_formal_2022_national_audit_v100"
), (
    "O RAIS_FORMAL_AUDIT_LOCK.json foi sobrescrito "
    "por outra execução."
)

assert audit_lock.get("status") == "AUDIT_PASSED"
assert audit_lock.get("critical_failures") == []


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


expected_path = Path(
    audit_lock["artifacts"]["source_audit"]
)

expected_hash = audit_lock[
    "artifact_hashes"
]["source_audit"]

print("\nCaminho registrado no lock:")
print(expected_path)

print("\nExiste agora:", expected_path.is_file())
print("Hash esperado:", expected_hash)

Mounted at /content/drive
Run ID do lock: rais_formal_2022_national_audit_v100
Status: AUDIT_PASSED
Critical failures: []

Caminho registrado no lock:
/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/rais_formal_certification/rais_source_schema_audit_rais_formal_2022_national_audit_v100.csv

Existe agora: True
Hash esperado: f1480f07d05f22c82f4b23e589b083f727b13128f1beeee85f67ba085be1353c


In [ ]:
OUTPUT_ROOT = (
    ROOT
    / "05_outputs"
)

candidates = sorted(
    OUTPUT_ROOT.rglob(
        "rais_source_schema_audit_*.csv"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

print("Candidatos encontrados:", len(candidates))

for path in candidates[:20]:
    print(
        path,
        "|",
        path.stat().st_size,
        "bytes",
    )

Candidatos encontrados: 1
/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/rais_formal_certification/rais_source_schema_audit_rais_formal_2022_national_audit_v100.csv | 15034 bytes


In [ ]:
SOURCE_AUDIT_PATH = None

# 1. Usa o caminho registrado no lock.
if expected_path.is_file():
    SOURCE_AUDIT_PATH = expected_path

# 2. Procura o mesmo run_id.
if SOURCE_AUDIT_PATH is None:
    run_id = audit_lock["run_id"]

    matching_name = [
        path
        for path in candidates
        if run_id in path.name
    ]

    if matching_name:
        SOURCE_AUDIT_PATH = matching_name[0]

# 3. Procura pelo SHA-256 registrado.
if SOURCE_AUDIT_PATH is None:
    for path in candidates:
        try:
            if sha256_file(path) == expected_hash:
                SOURCE_AUDIT_PATH = path
                break
        except OSError:
            pass

if SOURCE_AUDIT_PATH is None:
    print(
        "\nO CSV não foi localizado no diretório de outputs."
    )
else:
    print("\nArquivo recuperado:")
    print(SOURCE_AUDIT_PATH)

    actual_hash = sha256_file(
        SOURCE_AUDIT_PATH
    )

    print("Hash encontrado:", actual_hash)
    print("Hash esperado:  ", expected_hash)

    assert actual_hash == expected_hash, (
        "O arquivo encontrado não corresponde "
        "ao artefato certificado no lock."
    )

    source_audit = pd.read_csv(
        SOURCE_AUDIT_PATH
    )

    print("\nDimensão:", source_audit.shape)
    print("Colunas:", source_audit.columns.tolist())

    display(source_audit)

    print(
        "\nSOURCE AUDIT FOUND AND HASH VERIFIED"
    )


Arquivo recuperado:
/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/rais_formal_certification/rais_source_schema_audit_rais_formal_2022_national_audit_v100.csv
Hash encontrado: f1480f07d05f22c82f4b23e589b083f727b13128f1beeee85f67ba085be1353c
Hash esperado:   f1480f07d05f22c82f4b23e589b083f727b13128f1beeee85f67ba085be1353c

Dimensão: (7, 15)
Colunas: ['source_path', 'source_sha256', 'source_extension', 'source_size_bytes', 'inferred_year', 'encoding', 'delimiter', 'header_columns', 'canonical_mapping', 'rows_total', 'rows_active_all_occupations', 'rows_target_all', 'rows_target_active', 'status', 'notes']


,source_path,source_sha256,source_extension,source_size_bytes,inferred_year,encoding,delimiter,header_columns,canonical_mapping,rows_total,rows_active_all_occupations,rows_target_all,rows_target_active,status,notes
0,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,bd2dacaf12d898d3d4539da8dec30ba3126dab77e1daff...,.txt,7002566760,2022,latin1,;,"['Bairros SP', 'Bairros Fortaleza', 'Bairros R...","{'cbo2002': 'CBO Ocupação 2002', 'active_3112'...",14349520,9274065,22396,13782,PARSED,NaN
1,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,3035ca7e34f6a1c22b97015e84d3e060c8894f406d59e3...,.txt,10921853360,2022,latin1,;,"['Bairros SP', 'Bairros Fortaleza', 'Bairros R...","{'cbo2002': 'CBO Ocupação 2002', 'active_3112'...",22380845,14891791,64069,39687,PARSED,NaN
2,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,5e4ffeb55cd35534998fda23e8387c17ebb35d6ddb4004...,.txt,3641375504,2022,latin1,;,"['Bairros SP', 'Bairros Fortaleza', 'Bairros R...","{'cbo2002': 'CBO Ocupação 2002', 'active_3112'...",7461833,4850100,34917,19723,PARSED,NaN
3,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,f964c8f13058bd995d1c4a66e3a9c306d22c22a20b35bb...,.txt,1141944,2022,latin1,;,"['Bairros SP', 'Bairros Fortaleza', 'Bairros R...","{'cbo2002': 'CBO Ocupação 2002', 'active_3112'...",2338,1925,1,1,PARSED,NaN
4,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,396ca7ec2e5ad69462633bfba360c0ce75c4db4eb01ad1...,.txt,6629461968,2022,latin1,;,"['Bairros SP', 'Bairros Fortaleza', 'Bairros R...","{'cbo2002': 'CBO Ocupação 2002', 'active_3112'...",13584961,9777008,40469,27198,PARSED,NaN
5,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,5eaafbb53fe0febed183aeca33440cd09a9d49e23fac53...,.txt,2203446416,2022,latin1,;,"['Bairros SP', 'Bairros Fortaleza', 'Bairros R...","{'cbo2002': 'CBO Ocupação 2002', 'active_3112'...",4515257,3118437,12101,7553,PARSED,NaN
6,/content/drive/MyDrive/aCidadeAlgoritmica/SPIN...,5434f2efa722ec013668dfd94fa29930292f76986afcb3...,.txt,7902534408,2022,latin1,;,"['Bairros SP', 'Bairros Fortaleza', 'Bairros R...","{'cbo2002': 'CBO Ocupação 2002', 'active_3112'...",16193716,10877538,59142,36856,PARSED,NaN



SOURCE AUDIT FOUND AND HASH VERIFIED


In [ ]:
assert len(source_audit) == 7, (
    f"Esperadas 7 fontes; encontradas "
    f"{len(source_audit)}."
)

assert set(
    pd.to_numeric(
        source_audit["inferred_year"],
        errors="coerce",
    )
    .dropna()
    .astype(int)
) == {2022}

print(
    source_audit["status"]
    .value_counts(dropna=False)
)

allowed_status = {
    "PARSED",
    "NO_TARGET_ROWS",
}

invalid_status = source_audit[
    ~source_audit["status"].isin(
        allowed_status
    )
]

assert invalid_status.empty, (
    "Há fontes com status inesperado:\n"
    f"{invalid_status[['source_path', 'status']]}"
)

numeric_columns = [
    "rows_total",
    "rows_active_all_occupations",
    "rows_target_all",
    "rows_target_active",
]

for column in numeric_columns:
    source_audit[column] = pd.to_numeric(
        source_audit[column],
        errors="coerce",
    ).fillna(0)

summary = {
    column: int(source_audit[column].sum())
    for column in numeric_columns
}

print("\nResumo nacional:")
for key, value in summary.items():
    print(f"{key}: {value:,}")

assert summary["rows_total"] > 0
assert summary["rows_target_all"] > 0
assert summary["rows_target_active"] > 0

print("\nRAIS 2022 NATIONAL PREFLIGHT PASSED")

status
PARSED    7
Name: count, dtype: int64

Resumo nacional:
rows_total: 78,488,470
rows_active_all_occupations: 52,790,864
rows_target_all: 233,095
rows_target_active: 144,800

RAIS 2022 NATIONAL PREFLIGHT PASSED


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

RAIS_SCRIPT = (
    ROOT
    / "scripts"
    / "SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py"
)

assert RAIS_SCRIPT.is_file(), RAIS_SCRIPT

# source_audit já foi carregado na etapa anterior.
assert len(source_audit) == 7
assert (source_audit["status"] == "PARSED").all()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


exact_sources = []

for row in source_audit.itertuples(index=False):
    path = Path(row.source_path)

    assert path.is_file(), (
        f"Fonte auditada não encontrada: {path}"
    )

    actual_hash = sha256_file(path)

    assert actual_hash == row.source_sha256, (
        f"Hash divergente: {path}\n"
        f"Esperado: {row.source_sha256}\n"
        f"Encontrado: {actual_hash}"
    )

    exact_sources.append(path)

print("Fontes exatas validadas:", len(exact_sources))

for path in exact_sources:
    print(
        path.name,
        round(path.stat().st_size / 1024**3, 3),
        "GiB",
    )

print("\nSAME-SOURCE AUDIT → FULL GATE PASSED")

Fontes exatas validadas: 7
RAIS_VINC_PUB_SUL.txt 6.522 GiB
RAIS_VINC_PUB_SP.txt 10.172 GiB
RAIS_VINC_PUB_CENTRO_OESTE.txt 3.391 GiB
RAIS_VINC_PUB_NI.txt 0.001 GiB
RAIS_VINC_PUB_NORDESTE.txt 6.174 GiB
RAIS_VINC_PUB_NORTE.txt 2.052 GiB
RAIS_VINC_PUB_MG_ES_RJ.txt 7.36 GiB

SAME-SOURCE AUDIT → FULL GATE PASSED


In [ ]:
import subprocess
import sys

FULL_RUN_ID = "rais_formal_2022_national_full_v100"

EMPTY_DISCOVERY_ROOT = (
    ROOT
    / "00_admin"
    / "empty_roots"
    / FULL_RUN_ID
)

EMPTY_DISCOVERY_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

cmd_full = [
    sys.executable,
    str(RAIS_SCRIPT),
    "--root",
    str(ROOT),
    "--mode",
    "full",
    "--run-id",
    FULL_RUN_ID,
    "--years",
    "2022",
    "--rais-root",
    str(EMPTY_DISCOVERY_ROOT),
    "--primary-cbo",
    "519110",
    "--minimum-years",
    "1",
    "--chunksize",
    "300000",
]

for source_path in exact_sources:
    cmd_full.extend([
        "--source-file",
        str(source_path),
    ])

print("Executando certificação nacional completa:\n")
print(" ".join(cmd_full))
print()

process = subprocess.Popen(
    cmd_full,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(line, end="")

full_exit_code = process.wait()

print(
    "\nFull certification exit code:",
    full_exit_code,
)

Executando certificação nacional completa:

/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode full --run-id rais_formal_2022_national_full_v100 --years 2022 --rais-root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/empty_roots/rais_formal_2022_national_full_v100 --primary-cbo 519110 --minimum-years 1 --chunksize 300000 --source-file /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_intermediate/rais_formal_certification/rais_formal_2022_national_audit_v100/extracted_7z/02b6baa6d3f749f5/RAIS_VINC_PUB_SUL.txt --source-file /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_intermediate/rais_formal_certification/rais_formal_2022_national_audit_v100/extracted_7z/0ffdc75b2076bc2c/RAIS_VINC_PUB_SP.txt --source-file /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_intermediate/rais_formal_certification/rais_formal_20

In [ ]:
import json
from pathlib import Path

CERTIFICATION_LOCK = (
    ROOT
    / "00_admin"
    / "RAIS_FORMAL_CERTIFICATION_LOCK.json"
)

CORE_FREEZE = (
    ROOT
    / "00_admin"
    / "RAIS_FORMAL_CORE_FREEZE.json"
)

assert CERTIFICATION_LOCK.is_file(), (
    f"Lock não encontrado: {CERTIFICATION_LOCK}"
)

rais_lock = json.loads(
    CERTIFICATION_LOCK.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        rais_lock,
        ensure_ascii=False,
        indent=2,
    )
)

assert rais_lock["run_id"] == FULL_RUN_ID
assert rais_lock["mode"] == "full"
assert rais_lock["status"] == "CORE_CERTIFIED"
assert rais_lock["critical_failures"] == []
assert rais_lock["years_certified"] == [2022]
assert rais_lock["evidence_tier"] == "D"
assert rais_lock["platform_direct_observed"] is False

# Consistência exata entre audit e full.
assert rais_lock["n_target_links_all"] == 233095
assert rais_lock["n_active_primary_links"] == 144800

assert CORE_FREEZE.is_file(), (
    "O freeze do núcleo RAIS não foi criado."
)

print(
    "\nRAIS 2022 NATIONAL CORE "
    "CERTIFIED AND FROZEN"
)

NameError: name 'ROOT' is not defined

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

CERTIFICATION_LOCK = (
    ROOT
    / "00_admin"
    / "RAIS_FORMAL_CERTIFICATION_LOCK.json"
)

assert CERTIFICATION_LOCK.is_file(), (
    f"Lock não encontrado: {CERTIFICATION_LOCK}"
)

rais_lock = json.loads(
    CERTIFICATION_LOCK.read_text(encoding="utf-8")
)

print("Run ID:", rais_lock["run_id"])
print("Status:", rais_lock["status"])
print("Critical failures:", rais_lock["critical_failures"])
print("Anos:", rais_lock["years_certified"])
print("Vínculos-alvo:", rais_lock["n_target_links_all"])
print("Ativos primários:", rais_lock["n_active_primary_links"])

assert rais_lock["run_id"] == (
    "rais_formal_2022_national_full_v100"
)
assert rais_lock["status"] == "CORE_CERTIFIED"
assert rais_lock["critical_failures"] == []
assert rais_lock["years_certified"] == [2022]
assert rais_lock["n_target_links_all"] == 233095
assert rais_lock["n_active_primary_links"] == 144800
assert rais_lock["evidence_tier"] == "D"
assert rais_lock["platform_direct_observed"] is False

print("\nRAIS 2022 CORE CERTIFIED")

AssertionError: Lock não encontrado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/RAIS_FORMAL_CERTIFICATION_LOCK.json

In [3]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import time

drive.mount("/content/drive", force_remount=False)

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

ADMIN = ROOT / "00_admin"
ADMIN.mkdir(parents=True, exist_ok=True)

RUN_ID = "rais_formal_2022_national_full_v100"

LOCK_PATH = (
    ADMIN
    / "RAIS_FORMAL_CERTIFICATION_LOCK.json"
)

FREEZE_PATH = (
    ADMIN
    / "RAIS_FORMAL_CORE_FREEZE.json"
)

TABLE_DIR = (
    ROOT
    / "05_outputs"
    / "tables"
    / "rais_formal_certification"
)

REPORT_DIR = (
    ROOT
    / "06_reports"
    / "rais_formal_certification"
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def json_write(path: Path, obj: dict) -> None:
    path.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )


# Força a sincronização visível da montagem.
os.sync()
time.sleep(3)

print("Lock esperado:", LOCK_PATH)
print("Existe:", LOCK_PATH.is_file())

print("\nFreeze esperado:", FREEZE_PATH)
print("Existe:", FREEZE_PATH.is_file())

# Procura somente dentro do projeto, não no Drive inteiro.
found_locks = list(
    ROOT.rglob("RAIS_FORMAL_CERTIFICATION_LOCK.json")
)

found_freezes = list(
    ROOT.rglob("RAIS_FORMAL_CORE_FREEZE.json")
)

print("\nLocks encontrados no projeto:")
for path in found_locks:
    print(" -", path)

print("\nFreezes encontrados no projeto:")
for path in found_freezes:
    print(" -", path)

# Recupera de outro local, caso exista uma cópia válida.
if not LOCK_PATH.is_file() and found_locks:
    source = found_locks[0]

    if source.resolve() != LOCK_PATH.resolve():
        shutil.copy2(source, LOCK_PATH)
        print("\nLock copiado de:", source)

if not FREEZE_PATH.is_file() and found_freezes:
    source = found_freezes[0]

    if source.resolve() != FREEZE_PATH.resolve():
        shutil.copy2(source, FREEZE_PATH)
        print("Freeze copiado de:", source)


artifacts = {
    "inventory": (
        TABLE_DIR
        / f"rais_source_inventory_{RUN_ID}.csv"
    ),
    "source_audit": (
        TABLE_DIR
        / f"rais_source_schema_audit_{RUN_ID}.csv"
    ),
    "quality": (
        TABLE_DIR
        / f"rais_formal_quality_profile_{RUN_ID}.csv"
    ),
    "geography": (
        TABLE_DIR
        / f"rais_formal_annual_geography_{RUN_ID}.csv"
    ),
    "special_geographies": (
        TABLE_DIR
        / f"rais_formal_brasil_regiao_pe_recife_{RUN_ID}.csv"
    ),
    "demographics": (
        TABLE_DIR
        / f"rais_formal_demographic_profile_{RUN_ID}.csv"
    ),
    "golden": (
        TABLE_DIR
        / f"rais_formal_golden_tests_{RUN_ID}.csv"
    ),
    "target_all_parquet": (
        TABLE_DIR
        / f"rais_formal_target_links_all_{RUN_ID}.parquet"
    ),
    "active_primary_parquet": (
        TABLE_DIR
        / f"rais_formal_active_primary_links_{RUN_ID}.parquet"
    ),
    "report": (
        REPORT_DIR
        / f"rais_formal_certification_report_{RUN_ID}.md"
    ),
}

expected_hashes = {
    "inventory": (
        "273a91007eba0ad635acff9797f600d589"
        "fbcb4b65eb4c36f58bd582e1d5ba3b"
    ),
    "source_audit": (
        "f1480f07d05f22c82f4b23e589b083f7"
        "27b13128f1beeee85f67ba085be1353c"
    ),
    "quality": (
        "a169faef40b5f61a4a79d25aa9564382"
        "0ca0d22d3a818677b8d4bbf2ef1a8524"
    ),
    "geography": (
        "f617041eeebe02534b138a0639e693a97"
        "7b006e875d12c6ed4bd0c4cf14d1f2b"
    ),
    "special_geographies": (
        "dbb748f1328495135e1cfead7c1aa587"
        "9928f9b7525df0c40a6e7b7384346591"
    ),
    "demographics": (
        "9f6e7ee5625dd9b25100f352e3bd1d73"
        "c0224430466cc45dd0827f0769611add"
    ),
    "golden": (
        "01ba4719c80b6fe911b091a7c05124b6"
        "4eeece964e09c058ef8f9805daca546b"
    ),
    "target_all_parquet": (
        "e1a12332bb0e48635fe09a5d6e810b92"
        "a9449852943d85d24cf2fbf54c8a4ecf"
    ),
    "active_primary_parquet": (
        "80cd9aeaa639d890001e82fc8551a19aa"
        "7809cf43cf446445a14ffbb91c7066d"
    ),
    "report": (
        "563914cd7b5eaafd011cd47cc2da5fff2"
        "f7afc400e3a45a6b8722081206f99f7"
    ),
}

print("\nVERIFICAÇÃO DOS ARTEFATOS CERTIFICADOS")

verification_failures = []

for name, path in artifacts.items():
    exists = path.is_file()

    print(f"\n{name}")
    print(" Caminho:", path)
    print(" Existe:", exists)

    if not exists:
        verification_failures.append(
            f"{name}: arquivo ausente"
        )
        continue

    actual_hash = sha256_file(path)
    expected_hash = expected_hashes[name]

    print(" Hash esperado:  ", expected_hash)
    print(" Hash encontrado:", actual_hash)
    print(" Coincide:", actual_hash == expected_hash)

    if actual_hash != expected_hash:
        verification_failures.append(
            f"{name}: hash divergente"
        )

assert not verification_failures, (
    "A recuperação foi bloqueada porque existem "
    "problemas nos artefatos:\n"
    + "\n".join(verification_failures)
)

print(
    "\nTODOS OS ARTEFATOS CERTIFICADOS "
    "EXISTEM E POSSUEM OS HASHES ESPERADOS"
)

# Reconstrói o lock somente quando necessário.
if not LOCK_PATH.is_file():
    certification_lock = {
        "run_id": RUN_ID,
        "script_version": "1.0.0",
        "schema_version": (
            "spine-gpe-v7-rais-formal-"
            "certifier-1.0.0"
        ),
        "mode": "full",
        "status": "CORE_CERTIFIED",
        "critical_failures": [],
        "warnings": [
            {
                "test_id": "rais.real_income",
                "severity": "medium",
                "message": (
                    "Deflator não fornecido; "
                    "remuneração real não foi criada."
                ),
                "observed": None,
                "expected": (
                    "--deflator-csv for real income"
                ),
            },
            {
                "test_id": "golden.external",
                "severity": "medium",
                "message": (
                    "Golden externo oficial não fornecido; "
                    "certificação baseada em schema, universo "
                    "e consistência interna."
                ),
                "observed": None,
                "expected": (
                    "official golden CSV optional"
                ),
            },
        ],
        "years_requested": [2022],
        "years_certified": [2022],
        "primary_cbo_codes": ["519110"],
        "sensitivity_cbo_codes": [],
        "unit_of_analysis": "formal_employment_link",
        "primary_universe": (
            "target CBO links active on 31/12"
        ),
        "n_target_links_all": 233095,
        "n_active_primary_links": 144800,
        "evidence_tier": "D",
        "platform_direct_observed": False,
        "deflator_csv": None,
        "real_base_year": None,
        "depara_2024": None,
        "golden_csv": None,
        "claim_ceiling": (
            "Baseline administrativo de vínculos formais "
            "registrados no CBO-alvo; não representa "
            "informalidade, não identifica uso de plataforma "
            "e não equivale a trabalhadores únicos quando "
            "uma pessoa possui múltiplos vínculos."
        ),
        "artifacts": {
            name: str(path)
            for name, path in artifacts.items()
        },
        "artifact_hashes": expected_hashes,
        "created_at_utc": (
            "2026-07-26T05:04:02.556289+00:00"
        ),
    }

    json_write(
        LOCK_PATH,
        certification_lock,
    )

    print("\nLock administrativo recuperado:")
    print(LOCK_PATH)

# Valida o lock existente ou recuperado.
rais_lock = json.loads(
    LOCK_PATH.read_text(encoding="utf-8")
)

assert rais_lock["run_id"] == RUN_ID
assert rais_lock["status"] == "CORE_CERTIFIED"
assert rais_lock["critical_failures"] == []
assert rais_lock["years_certified"] == [2022]
assert rais_lock["n_target_links_all"] == 233095
assert rais_lock["n_active_primary_links"] == 144800

lock_hash = sha256_file(LOCK_PATH)

# Reconstrói o freeze somente quando necessário.
if not FREEZE_PATH.is_file():
    freeze = {
        "freeze_id": RUN_ID,
        "status": "FROZEN",
        "component": "RAIS_FORMAL_BASELINE",
        "certification_lock": str(LOCK_PATH),
        "certification_lock_sha256": lock_hash,
        "active_primary_parquet": str(
            artifacts["active_primary_parquet"]
        ),
        "active_primary_parquet_sha256": (
            expected_hashes["active_primary_parquet"]
        ),
        "special_geographies": str(
            artifacts["special_geographies"]
        ),
        "special_geographies_sha256": (
            expected_hashes["special_geographies"]
        ),
        "read_only": True,
        "evidence_tier": "D",
        "platform_direct_observed": False,
        "claim_ceiling": (
            rais_lock["claim_ceiling"]
        ),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    json_write(
        FREEZE_PATH,
        freeze,
    )

    print("Freeze administrativo recuperado:")
    print(FREEZE_PATH)

freeze = json.loads(
    FREEZE_PATH.read_text(encoding="utf-8")
)

assert freeze["freeze_id"] == RUN_ID
assert freeze["status"] == "FROZEN"
assert freeze["read_only"] is True
assert (
    freeze["certification_lock_sha256"]
    == sha256_file(LOCK_PATH)
)
assert (
    freeze["active_primary_parquet_sha256"]
    == sha256_file(
        artifacts["active_primary_parquet"]
    )
)
assert (
    freeze["special_geographies_sha256"]
    == sha256_file(
        artifacts["special_geographies"]
    )
)

# Memorando de recuperação, separado do lock.
RECOVERY_MEMO = (
    ADMIN
    / "RAIS_FORMAL_ADMIN_LOCK_RECOVERY.json"
)

recovery_memo = {
    "event": (
        "administrative_lock_recovery_after_"
        "successful_certified_run"
    ),
    "run_id": RUN_ID,
    "certification_status": "CORE_CERTIFIED",
    "certification_lock": str(LOCK_PATH),
    "certification_lock_sha256": (
        sha256_file(LOCK_PATH)
    ),
    "core_freeze": str(FREEZE_PATH),
    "core_freeze_sha256": (
        sha256_file(FREEZE_PATH)
    ),
    "artifact_hashes_verified": True,
    "artifacts_verified": sorted(
        artifacts.keys()
    ),
    "estimations_recomputed": False,
    "reason": (
        "O lock administrativo não estava visível "
        "na montagem do Google Drive após o run "
        "certificado. O lock e o freeze foram "
        "recuperados exclusivamente após verificar "
        "todos os artefatos e hashes impressos pelo "
        "run original."
    ),
    "recovered_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

json_write(
    RECOVERY_MEMO,
    recovery_memo,
)

os.sync()

print("\n" + "=" * 72)
print("RAIS 2022 NATIONAL CORE CERTIFIED AND FROZEN")
print("=" * 72)
print("Lock:", LOCK_PATH)
print("Lock SHA-256:", sha256_file(LOCK_PATH))
print("Freeze:", FREEZE_PATH)
print("Freeze SHA-256:", sha256_file(FREEZE_PATH))
print("Recovery memo:", RECOVERY_MEMO)

Mounted at /content/drive
Lock esperado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/RAIS_FORMAL_CERTIFICATION_LOCK.json
Existe: True

Freeze esperado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/RAIS_FORMAL_CORE_FREEZE.json
Existe: True

Locks encontrados no projeto:
 - /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/RAIS_FORMAL_CERTIFICATION_LOCK.json

Freezes encontrados no projeto:
 - /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/RAIS_FORMAL_CORE_FREEZE.json

VERIFICAÇÃO DOS ARTEFATOS CERTIFICADOS

inventory
 Caminho: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/rais_formal_certification/rais_source_inventory_rais_formal_2022_national_full_v100.csv
 Existe: True
 Hash esperado:   273a91007eba0ad635acff9797f600d589fbcb4b65eb4c36f58bd582e1d5ba3b
 Hash encontrado: 273a91007eba0ad635acff9797f600d589fbcb4b65eb4c36f58bd582e1d5ba3b
 Coincide: True

source_audit
 Caminho: /content/drive/MyDrive

In [4]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

LOCK_PATH = (
    ROOT
    / "00_admin"
    / "RAIS_FORMAL_CERTIFICATION_LOCK.json"
)

rais_lock = json.loads(
    LOCK_PATH.read_text(encoding="utf-8")
)

quality = pd.read_csv(
    rais_lock["artifacts"]["quality"]
)

special = pd.read_csv(
    rais_lock["artifacts"]["special_geographies"]
)

geography = pd.read_csv(
    rais_lock["artifacts"]["geography"]
)

demographics = pd.read_csv(
    rais_lock["artifacts"]["demographics"]
)

print("QUALITY PROFILE")
display(quality)

print("\nBRASIL, REGIÕES, PERNAMBUCO E RECIFE")
display(special)

print("\nGEOGRAFIAS — PRIMEIRAS LINHAS")
display(geography.head(100))

print("\nPERFIS DEMOGRÁFICOS")
display(demographics.head(200))

QUALITY PROFILE


,year,n_target_links_all_year,n_target_links_active_3112,active_share_within_target,missing_municipality_rate_active,missing_income_rate_active,nonpositive_income_rate_active,invalid_hours_rate_active,recife_links_active,pe_links_active,n_source_files
0,2022,233095,144800,0.621206,0.0,0.0,0.093715,0.000546,1984,5215,7



BRASIL, REGIÕES, PERNAMBUCO E RECIFE


,year,special_geography,n_links_active,n_source_files,income_monthly_nominal_mean,income_monthly_nominal_median,income_hour_nominal_mean,income_hour_nominal_median,contract_hours_mean,contract_hours_median,missing_income_rate,missing_hours_rate
0,2022,Brasil,144800,7,1707.781195,1717.25,9.664224,9.247829,42.193854,44.0,0.0,0.0
1,2022,Nordeste,27385,7,1590.135801,1623.55,8.753766,8.516843,43.036808,44.0,0.0,0.0
2,2022,Pernambuco,5215,5,1587.231990,1653.99,8.947624,8.673920,42.940748,44.0,0.0,0.0
3,2022,Recife,1984,4,1605.069582,1663.73,9.320949,8.743073,42.637601,44.0,0.0,0.0



GEOGRAFIAS — PRIMEIRAS LINHAS


,geography_level,year,geography,n_links_active,n_source_files,income_monthly_nominal_mean,income_monthly_nominal_median,income_hour_nominal_mean,income_hour_nominal_median,contract_hours_mean,contract_hours_median,missing_income_rate,missing_hours_rate,region,uf,municipality6
0,BRASIL,2022,Brasil,144800,7,1707.781195,1717.25,9.664224,9.247829,42.193854,44.0,0.0,0.0,NaN,NaN,NaN
1,REGION,2022,NaN,19731,6,1618.499020,1660.36,8.738596,8.768647,43.112665,44.0,0.0,0.0,Centro-Oeste,NaN,NaN
2,REGION,2022,NaN,27385,7,1590.135801,1623.55,8.753766,8.516843,43.036808,44.0,0.0,0.0,Nordeste,NaN,NaN
3,REGION,2022,NaN,7736,6,1704.918544,1668.38,9.254560,8.767706,43.551060,44.0,0.0,0.0,Norte,NaN,NaN
4,REGION,2022,NaN,75900,6,1750.541179,1785.00,10.110091,9.683623,41.759223,44.0,0.0,0.0,Sudeste,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,MUNICIPALITY,2022,NaN,29,1,1611.203448,1619.37,8.523508,8.470394,42.068966,44.0,0.0,0.0,NaN,AL,270670.0
96,MUNICIPALITY,2022,NaN,3,1,1544.126667,1599.22,8.076821,8.364996,44.000000,44.0,0.0,0.0,NaN,AL,270680.0
97,MUNICIPALITY,2022,NaN,9,1,1537.974444,1562.77,8.044641,8.174338,44.000000,44.0,0.0,0.0,NaN,AL,270690.0
98,MUNICIPALITY,2022,NaN,2,1,1555.140000,1555.14,8.134428,8.134428,44.000000,44.0,0.0,0.0,NaN,AL,270720.0



PERFIS DEMOGRÁFICOS


,year,dimension,category,n_links_active,share_within_year,income_hour_nominal_mean,contract_hours_mean
0,2022,sex,1,141317,0.975946,9.674185,42.199417
1,2022,sex,2,3483,0.024054,9.260043,41.968131
2,2022,race,1,187,0.001291,9.960627,42.855615
3,2022,race,2,45329,0.313046,10.429144,41.507512
4,2022,race,4,6514,0.044986,9.541030,42.332515
...,...,...,...,...,...,...,...
195,2022,cnae_class,26329,1,0.000007,9.962705,44.000000
196,2022,cnae_class,26400,4,0.000028,8.881672,43.000000
197,2022,cnae_class,26515,6,0.000041,19.064878,44.000000
198,2022,cnae_class,26701,1,0.000007,9.801548,44.000000


In [5]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

LOCK_PATH = (
    ROOT
    / "00_admin"
    / "RAIS_FORMAL_CERTIFICATION_LOCK.json"
)

rais_lock = json.loads(
    LOCK_PATH.read_text(encoding="utf-8")
)

ACTIVE_PATH = Path(
    rais_lock["artifacts"]["active_primary_parquet"]
)

assert ACTIVE_PATH.is_file(), ACTIVE_PATH

active = pd.read_parquet(ACTIVE_PATH)

print("Linhas:", len(active))
print("Colunas:", active.columns.tolist())

assert len(active) == 144800

for column in [
    "income_monthly_nominal",
    "income_hour_nominal",
    "contract_hours",
]:
    active[column] = pd.to_numeric(
        active[column],
        errors="coerce",
    )

active["income_status"] = np.select(
    [
        active["income_monthly_nominal"].isna(),
        active["income_monthly_nominal"].lt(0),
        active["income_monthly_nominal"].eq(0),
        active["income_monthly_nominal"].gt(0),
    ],
    [
        "MISSING",
        "NEGATIVE",
        "ZERO",
        "POSITIVE",
    ],
    default="OTHER",
)

active["valid_hours"] = (
    active["contract_hours"].between(1, 100)
)

active["valid_monthly_income"] = (
    active["income_monthly_nominal"].gt(0)
)

active["valid_hourly_income"] = (
    active["valid_monthly_income"]
    & active["valid_hours"]
    & active["income_hour_nominal"].notna()
)

print("\nSTATUS DA RENDA")
display(
    active["income_status"]
    .value_counts(dropna=False)
    .rename_axis("income_status")
    .reset_index(name="n_links")
    .assign(
        share=lambda df: df["n_links"] / len(active)
    )
)

Linhas: 144800
Colunas: ['cbo2002', 'active_3112', 'municipality_work', 'income_avg_nominal', 'income_dec_nominal', 'income_avg_sm', 'contract_hours', 'sex', 'race', 'education', 'age', 'age_group', 'link_type', 'cnae_class', 'admission_type', 'establishment_size', 'legal_nature', 'year', 'municipality_work_raw', 'municipality6', 'uf_code', 'uf', 'region', 'is_pe', 'is_recife', 'income_monthly_nominal', 'income_source', 'income_hour_nominal', 'record_quality_valid_hours', 'record_quality_positive_income', 'record_quality_valid_municipality', 'source_sha256', 'source_file', 'cbo_scope']

STATUS DA RENDA


,income_status,n_links,share
0,POSITIVE,131230,0.906285
1,ZERO,13570,0.093715


In [6]:
geographies = {
    "Brasil": pd.Series(
        True,
        index=active.index,
    ),
    "Nordeste": active["region"].eq("Nordeste"),
    "Pernambuco": active["uf"].eq("PE"),
    "Recife": active["is_recife"].fillna(False),
}

rows = []

for geography, mask in geographies.items():
    sub = active.loc[mask].copy()

    positive_monthly = sub.loc[
        sub["valid_monthly_income"]
    ]

    positive_hourly = sub.loc[
        sub["valid_hourly_income"]
    ]

    rows.append({
        "geography": geography,

        "n_active_all": int(len(sub)),

        "n_income_positive": int(
            len(positive_monthly)
        ),

        "n_income_nonpositive": int(
            (~sub["valid_monthly_income"])
            .sum()
        ),

        "nonpositive_income_share": float(
            (~sub["valid_monthly_income"])
            .mean()
        ),

        "n_hourly_analytical": int(
            len(positive_hourly)
        ),

        # Universo atualmente congelado
        "income_monthly_mean_all_active": float(
            sub["income_monthly_nominal"].mean()
        ),

        "income_hour_mean_all_active": float(
            sub["income_hour_nominal"].mean()
        ),

        # Universo recomendado para renda
        "income_monthly_mean_positive": float(
            positive_monthly[
                "income_monthly_nominal"
            ].mean()
        ),

        "income_monthly_median_positive": float(
            positive_monthly[
                "income_monthly_nominal"
            ].median()
        ),

        "income_hour_mean_positive_valid_hours": float(
            positive_hourly[
                "income_hour_nominal"
            ].mean()
        ),

        "income_hour_median_positive_valid_hours": float(
            positive_hourly[
                "income_hour_nominal"
            ].median()
        ),

        "contract_hours_mean_analytical": float(
            positive_hourly[
                "contract_hours"
            ].mean()
        ),

        "contract_hours_median_analytical": float(
            positive_hourly[
                "contract_hours"
            ].median()
        ),

        "n_source_files": int(
            sub["source_sha256"].nunique()
        ),
    })

income_adjudication = pd.DataFrame(rows)

display(income_adjudication)

,geography,n_active_all,n_income_positive,n_income_nonpositive,nonpositive_income_share,n_hourly_analytical,income_monthly_mean_all_active,income_hour_mean_all_active,income_monthly_mean_positive,income_monthly_median_positive,income_hour_mean_positive_valid_hours,income_hour_median_positive_valid_hours,contract_hours_mean_analytical,contract_hours_median_analytical,n_source_files
0,Brasil,144800,131230,13570,0.093715,131172,1707.781195,9.664224,1884.376415,1762.595,10.662459,9.498509,42.286090,44.0,7
1,Nordeste,27385,25040,2345,0.085631,25025,1590.135801,8.753766,1739.052273,1642.620,9.571251,8.646877,43.147932,44.0,7
2,Pernambuco,5215,4763,452,0.086673,4759,1587.231990,8.947624,1737.857407,1680.030,9.793690,8.851815,43.201093,44.0,5
3,Recife,1984,1813,171,0.086190,1813,1605.069582,9.320949,1756.457832,1695.620,10.200090,8.927241,42.896856,44.0,4


In [7]:
OUTPUT_DIR = (
    ROOT
    / "05_outputs"
    / "tables"
    / "rais_formal_certification"
)

OUTPUT_PATH = (
    OUTPUT_DIR
    / (
        "rais_formal_income_universe_"
        "adjudication_2022_v100.csv"
    )
)

income_adjudication.to_csv(
    OUTPUT_PATH,
    index=False,
)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

print("Arquivo:", OUTPUT_PATH)
print("SHA-256:", sha256_file(OUTPUT_PATH))

Arquivo: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/rais_formal_certification/rais_formal_income_universe_adjudication_2022_v100.csv
SHA-256: 0ef2e207212aa7f71eba6ae0b9d3e1a7f36e90b0ff3e08f2c705bbf0e045b23a
